<a href="https://colab.research.google.com/github/MottaDavide/MaskArchitectureAnomaly_CourseProject/blob/main/eomt/inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

© 2025 Mobile Perception Systems Lab at TU/e. All rights reserved. Licensed under the MIT License.

### Configure the loop to evaluate the model

Set EVAL_SINGLE to FALSE if you want to evaluate the COCO-trained or Cityscapes-trained model on the full validation set (cityscapes val).

You can select the model i the Setup. 

If you only want to visualize one picture, set EVAL_SINGLE to True

In [ ]:
# Evaluation configuration
# Select which classes to use for evaluation
# Select whether to run a loop or visualize a single image

# Cityscapes train_ids:
# 0=road, 1=sidewalk, 2=building, 3=wall, 4=fence, 5=pole,
# 6=traffic light, 7=traffic sign, 8=vegetation, 9=terrain,
# 10=sky, 11=person, 12=rider, 13=car, 14=truck, 15=bus,
# 16=train, 17=motorcycle, 18=bicycle

EXCLUDE_CLASSES = []       # [] = all classes. [3,4,5,7,9,12] = excluded classes in the evaluation
EVAL_SINGLE = False   # True = visualize one image (img_idx in setup) | False = all cityscape validation dataset



## Setup

In [ ]:
import os
from pathlib import Path
import zipfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab")
    PROJECT_ROOT = Path.cwd()
    DATA_ROOT = Path("/content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/datasets")
    WEIGHTS_ROOT = Path("/content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights")

else:
    print("Running locally")
    PROJECT_ROOT = Path.cwd()
    DATA_ROOT = PROJECT_ROOT / "datasets"
    WEIGHTS_ROOT = PROJECT_ROOT / "weights"

os.chdir(PROJECT_ROOT)

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("WEIGHTS_ROOT:", WEIGHTS_ROOT)


CITYSCAPES_ROOT = DATA_ROOT / "cityscapes"
COCO_ROOT = DATA_ROOT / "coco"

In [ ]:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib

seed_everything(0, verbose=False)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
# set the GPU, or "cpu" (slower)
# TODO: change to the GPU you want to use
img_idx = 5  # TODO: change to the index of the image you want to visualize

MODEL = "coco" # "coco" or "cityscapes"

if MODEL == "coco":
    config_path = "configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
    ckpt_path   = WEIGHTS_ROOT/"eomt_coco.bin"
else:
    config_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
    ckpt_path   = WEIGHTS_ROOT/"eomt_cityscapes.bin"

data_config_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml" 
data_path = DATA_ROOT /"cityscapes"
zip_path = DATA_ROOT / "coco" / "panoptic_annotations_trainval2017.zip"
# TODO: change to the dataset directory "/mnt/sda1/tkerssies" 
# change with the direcoty of datasets 

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

with open(data_config_path, "r") as f:
    data_config = yaml.safe_load(f)

def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

## Load dataset

Ensure the dataset files are correctly prepared and placed in the folder specified by `data_path`.

In [ ]:
data_module_name, class_name = "datasets.cityscapes_semantic.CityscapesSemantic".rsplit(".", 1)
#modified to  always use cityscapes dataset
# config["data"]["class_path"].rsplit(".", 1)

data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = {}
# modified 
# config["data"].get("init_args", {})

model_img_size = (640, 640) if MODEL == "coco" else (1024, 1024)

data = data_module(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    img_size=model_img_size,
    **data_module_kwargs
).setup()

model_num_classes = 133 if MODEL == "coco" else data.num_classes

## Load model

In [ ]:
warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=data.img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=model_num_classes,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}
if "stuff_classes" in config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]

model = (
    lit_cls(
        img_size=data.img_size,
        num_classes=model_num_classes,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)

## Load pre-trained weights from Hugging Face Hub
The model weights are downloaded from the Hugging Face Hub using the logger name from the config. Make sure you have a working internet connection.

In [ ]:
# state_dict = torch.load(ckpt_path, map_location=device, weights_only=True)
# model.load_state_dict(state_dict, strict=False)


if os.path.exists(ckpt_path):
    state_dict = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict, strict=False)
    print("Pesi caricati localmente da:", ckpt_path)
else:
    name = config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")

    if name is None:
        warnings.warn("No logger name found in the config. Please specify a model name.")
    else:
        try:
            state_dict_path = hf_hub_download(
                repo_id=f"tue-mps/{name}",
                filename="pytorch_model.bin",
            )

            is_dinov3 = "dinov3" in name

            if is_dinov3:
                model_kwargs["ckpt_path"] = state_dict_path
                model_kwargs["delta_weights"] = True

            model = (
                lit_cls(
                    img_size=data.img_size,
                    num_classes=data.num_classes,
                    network=network,
                    **model_kwargs,
                )
                .eval()
                .to(device)
            )

            if not is_dinov3:
                state_dict = torch.load(
                    state_dict_path, map_location=f"cuda:{device}", weights_only=True
                )
                model.load_state_dict(state_dict, strict=False)

        except RepositoryNotFoundError:
            warnings.warn(
                f"Pre-trained model not found for `{name}`. Please load your own checkpoint."
            )

## Semantic inference (pixel-wise classification)

> This inference method also works when applied to a model trained for panoptic segmentation.

Semantic inference computes per-pixel class scores by combining mask and class predictions:

$$
\sum_i p_i(c) \cdot m_i[h, w]
$$

Here, $p_i(c)$ is the class probability for class $c$ (excluding "no object"), and $m_i[h, w]$ is the sigmoid-normalized mask value for query $i$ at pixel $(h, w)$. The final class is selected by taking the argmax over classes.  
  
*This inference method was originally introduced in MaskFormer.*

In [ ]:
IGNORE_INDEX = 255


def infer_semantic(img, target):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], data.img_size, mode="bilinear"
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()
    target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[
        0
    ].numpy()
    return pred_array, target_array


def plot_semantic_results(img, pred_array, target_array):
    mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img.permute(1, 2, 0).cpu().numpy())
    axes[0].set_title("Image")
    axes[1].imshow(apply_colormap(pred_array, mapping))
    axes[1].set_title("Prediction")
    axes[2].imshow(apply_colormap(target_array, mapping))
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


if EVAL_SINGLE and MODEL == "cityscapes":
    img, target = data.val_dataloader().dataset[img_idx]
    pred_array, target_array = infer_semantic(img, target)
    plot_semantic_results(img, pred_array, target_array)

## Panoptic inference (segmentation with instance IDs)

> This inference method also works when applied to a model trained for instance segmentation.

Panoptic inference assigns each pixel $[h, w]$ to the query $i$ that maximizes the product of class and mask confidence:

$$
p_i(c_i) \cdot m_i[h, w]
$$

where $c_i = \arg\max_c \, p_i(c)$ is the most likely class for query $i$. A pixel is assigned to a query only if both the class confidence and mask confidence are high. Pixels assigned to the same query form a segment labeled with $c_i$. "Stuff" segments with the same class are merged; "thing" segments are kept distinct using the query index. Low-confidence and heavily occluded predictions are filtered out.  
  
*This inference method was originally introduced in MaskFormer.*

In [ ]:
def infer_panoptic(img, target):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(
            mask_logits, img_sizes
        )

        preds = model.to_per_pixel_preds_panoptic(
            mask_logits,
            class_logits_per_layer[-1],
            model.stuff_classes,
            model.mask_thresh,
            model.overlap_thresh,
        )[0].cpu()

    pred = preds.numpy()
    sem_pred, inst_pred = pred[..., 0], pred[..., 1]

    target_seg = model.to_per_pixel_targets_panoptic([target])[0].cpu().numpy()
    sem_target, inst_target = target_seg[..., 0], target_seg[..., 1]

    return sem_pred, inst_pred, sem_target, inst_target


def draw_black_border(sem, inst, mapping):
    h, w = sem.shape
    out = np.zeros((h, w, 3))
    for s in np.unique(sem):
        out[sem == s] = mapping[s]

    combined = sem.astype(np.int64) * 100000 + inst.astype(np.int64)
    border = np.zeros((h, w), dtype=bool)
    border[1:, :] |= combined[1:, :] != combined[:-1, :]
    border[:-1, :] |= combined[1:, :] != combined[:-1, :]
    border[:, 1:] |= combined[:, 1:] != combined[:, :-1]
    border[:, :-1] |= combined[:, 1:] != combined[:, :-1]
    out[border] = 0
    return out


def plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target):
    all_ids = np.union1d(np.unique(sem_pred), np.unique(sem_target))
    mapping = {
        s: (
            [0, 0, 0]
            if s == -1 or s == model.num_classes
            else plt.cm.hsv(i / len(all_ids))[:3]
        )
        for i, s in enumerate(all_ids)
    }

    vis_pred = draw_black_border(sem_pred, inst_pred, mapping)
    vis_target = draw_black_border(sem_target, inst_target, mapping)

    img_np = (
        img.cpu().numpy().transpose(1, 2, 0) if img.dim() == 3 else img.cpu().numpy()
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].set_title("Input")
    axes[1].imshow(vis_pred)
    axes[1].set_title("Prediction")
    axes[2].imshow(vis_target)
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


if EVAL_SINGLE and MODEL == "coco":
    img, target = data.val_dataloader().dataset[img_idx]
    sem_pred, inst_pred, sem_target, inst_target = infer_panoptic(img, target)
    plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target)

## Mapping COCO to Cityscapes

This will map the COCO classes to the Cityscapes classes in preparation for the evaluation loop (on the cityscapes validation set). 

In [ ]:
import pandas as pd
from torchvision.datasets import Cityscapes

rows = []

for cls in Cityscapes.classes:
    if not cls.ignore_in_eval:
        rows.append({
            "model_id": cls.train_id,
            "name": cls.name,
            "original_id": cls.id
        })

df_cs = pd.DataFrame(rows).sort_values("model_id")

df_cs.style.hide(axis="index")

In [ ]:
import json
import zipfile
from pathlib import Path
import pandas as pd

from datasets.coco_panoptic import CLASS_MAPPING


with zipfile.ZipFile(zip_path) as z:
    with z.open("annotations/panoptic_val2017.json") as f:
        coco_data  = json.load(f)


print("COCO PANOPTIC - id usati dal modello")
print("-" * 80)

rows = []

for cat in coco_data["categories"]:
    coco_original_id = cat["id"]
    name = cat["name"]
    isthing = cat["isthing"]

    if coco_original_id in CLASS_MAPPING:
        model_id = CLASS_MAPPING[coco_original_id]
        rows.append((model_id, coco_original_id, name, isthing))

rows = sorted(rows, key=lambda x: x[0])


df = pd.DataFrame(rows, columns=["model_id", "coco_id", "name", "isthing"])
df.style.hide(axis="index")


In [ ]:
if not EVAL_SINGLE:
    _COCO_CS_RAW = {
    100: 0,      # road
    123: 1,      # pavement-merged -> sidewalk
    91: 2,       # house -> building 
    # 101: 2,      # roof -> building    ????
    129: 2,      # building-other-merged -> building
    109: 3,      # wall-brick -> wall
    110: 3,      # wall-stone -> wall
    111: 3,      # wall-tile -> wall       
    112: 3,      # wall-wood -> wall       
    # 114: 3,    # window-blind -> wall      ??
    # 115: 3,    # window-other -> wall      ??
    131: 3,      # wall-other-merged -> wall
    117: 4,      # fence-merged -> fence
    9: 6,        # traffic light
    # 11: 7,       # stop sign -> traffic sign  ??
    # 88: 8,     # flower -> vegetation       ??
    116: 8,      # tree-merged -> vegetation
    # 97: 9,     # # playingfield -> terrain  ??
    # 102: 9,    # # sand -> terrain          ??
    125: 9,      # grass-merged -> terrain
    119: 10,     # sky-other-merged -> sky
    0: 11,       # person
    2: 13,       # car
    7: 14,       # truck
    5: 15,       # bus
    6: 16,       # train
    3: 17,       # motorcycle 
    1: 18,       # bicycle
    }

    COCO_TO_CS = torch.full((133,), -1, dtype=torch.long)
    for k, v in _COCO_CS_RAW.items():
        COCO_TO_CS[k] = v


## Inference 

In [ ]:
if not EVAL_SINGLE:
    def get_semantic_logits(img):
        with torch.no_grad():
            imgs = [img.to(device)]
            img_sizes = [img.shape[-2:]]
            crops, origins = model.window_imgs_semantic(imgs)
            ml, cl = model(crops)
            ml = F.interpolate(
                ml[-1],
                data.img_size,
                mode="bilinear"
            )
            crop_logits = model.to_per_pixel_logits_semantic(ml, cl[-1])
            logits = model.revert_window_logits_semantic(
                crop_logits,
                origins,
                img_sizes
            )[0]

        if MODEL == "coco":
            H, W = logits.shape[-2:]
            cs_logits = torch.full(
                (19, H, W),
                -float("inf"),
                device=logits.device,
                dtype=logits.dtype
            )
            for coco_id in range(133):
                cs_id = COCO_TO_CS[coco_id].item()
                if cs_id >= 0:
                    cs_logits[cs_id] = torch.maximum(
                        cs_logits[cs_id],
                        logits[coco_id]
                    )
            cs_logits = torch.nan_to_num(
                cs_logits,
                neginf=-1e9
            )
            return cs_logits.cpu()

        return logits.cpu()

In [ ]:
# if not EVAL_SINGLE:
#     def get_semantic_logits(img):
#         with torch.no_grad():
#             imgs = [img.to(device)]
#             img_sizes = [img.shape[-2:]]
#             crops, origins = model.window_imgs_semantic(imgs)
#             ml, cl = model(crops)
#             ml = F.interpolate(ml[-1], data.img_size, mode="bilinear")
#             crop_logits = model.to_per_pixel_logits_semantic(ml, cl[-1])
#             logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)[0]

#         if MODEL == "coco":
#             H, W = logits.shape[-2:]
#             cs_logits = torch.zeros(19, H, W)
#             for i in range(133):
#                 c = COCO_TO_CS[i].item()
#                 if c >= 0:
#                     cs_logits[c] += logits[i].cpu()
#             return cs_logits
#         return logits.cpu()


## Evaluation loop (looping on cityscapes validation dataset)

In [ ]:
from torchmetrics.classification import MulticlassJaccardIndex

CS_NAMES = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic light", "traffic sign", "vegetation", "terrain", "sky",
    "person", "rider", "car", "truck", "bus", "train",
    "motorcycle", "bicycle"
]

IGNORE_INDEX = 255

# Classi Cityscapes non rappresentabili bene dal modello COCO.
COCO_UNMAPPED_CS_CLASSES = [5, 7, 12]  # pole, traffic sign, rider

# if MODEL == "coco":
#     eval_exclude_classes = sorted(set(EXCLUDE_CLASSES + COCO_UNMAPPED_CS_CLASSES))
# else:
#     eval_exclude_classes = sorted(set(EXCLUDE_CLASSES))
eval_exclude_classes = sorted(
    set(EXCLUDE_CLASSES + COCO_UNMAPPED_CS_CLASSES)
)

dataset = data.val_dataloader().dataset

if EVAL_SINGLE:
    img, target = dataset[img_idx]
    if MODEL == "coco":
        sem_pred, inst_pred, sem_target, inst_target = infer_panoptic(img, target)
        plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target)
    else:
        pred_array, target_array = infer_semantic(img, target)
        plot_semantic_results(img, pred_array, target_array)

else:
    metric = MulticlassJaccardIndex(
        num_classes=19,
        ignore_index=IGNORE_INDEX,
        average=None
    )

    for i, idx in enumerate(range(len(dataset))):
        img, target = dataset[idx]

        gt = model.to_per_pixel_targets_semantic(
            [target],
            IGNORE_INDEX
        )[0].cpu()

        for cls in eval_exclude_classes:
            gt[gt == cls] = IGNORE_INDEX

        pred = get_semantic_logits(img).argmax(0).cpu()

        metric.update(
            pred.unsqueeze(0),
            gt.unsqueeze(0)
        )

        if (i + 1) % 50 == 0:
            print(f"{i + 1}/{len(dataset)} images processed")

    iou = metric.compute()


    valid = torch.ones(19, dtype=torch.bool)

    for c in eval_exclude_classes:
        valid[c] = False

    rows = []

    for i, name in enumerate(CS_NAMES):
        rows.append({
            "class_id": i,
            "class_name": name,
            "IoU (%)": iou[i].item() * 100,
            "evaluated": bool(valid[i]),
            "excluded": not bool(valid[i]),
        })

    df_iou = pd.DataFrame(rows)

    df_iou_valid = df_iou[df_iou["evaluated"]].copy()

    miou = df_iou_valid["IoU (%)"].mean()

    print(f"Model: {MODEL}")
    print(f"Excluded classes: {[CS_NAMES[c] for c in eval_exclude_classes]}")
    print(f"mIoU: {miou:.2f}%")

    df_report = df_iou_valid[["class_id", "class_name", "IoU (%)"]]

    df_report.style.hide(axis="index").format({
        "IoU (%)": "{:.2f}"
    })

In [ ]:
# from torchmetrics.classification import MulticlassJaccardIndex

# CS_NAMES = ["road","sidewalk","building","wall","fence","pole",
#             "traffic light","traffic sign","vegetation","terrain","sky",
#             "person","rider","car","truck","bus","train","motorcycle","bicycle"]

# dataset = data.val_dataloader().dataset

# if EVAL_SINGLE:
#     img, target = dataset[img_idx]
#     sem_pred, inst_pred, sem_target, inst_target = infer_panoptic(img, target)
#     plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target)

# else:
#     metric = MulticlassJaccardIndex(19, ignore_index=255, average=None)

#     for i, idx in enumerate(list(range(len(dataset)))):
#         img, target = dataset[idx]
#         gt = model.to_per_pixel_targets_semantic([target], 255)[0]
#         for cls in EXCLUDE_CLASSES:
#             gt[gt == cls] = 255
#         pred = get_semantic_logits(img).argmax(0)
#         metric.update(pred[None], gt[None])
#         if (i + 1) % 50 == 0:
#             print(f"  {i+1}/{len(dataset)} images processed")

#     iou = metric.compute()
#     valid = torch.ones(19, dtype=torch.bool)
#     for c in EXCLUDE_CLASSES:
#         valid[c] = False

#     print(f"\nModel: {MODEL}  |  Excluded: {[CS_NAMES[c] for c in EXCLUDE_CLASSES] or 'none'}\n")
#     print(f"{'Class':<20} {'IoU (%)':>8}")
#     print("-" * 30)
#     for i, name in enumerate(CS_NAMES):
#         if valid[i]:
#             print(f"{i:<3} {name:<20} {iou[i].item()*100:>7.1f}")
#     print("-" * 30)
#     print(f"{'mIoU':<20} {iou[valid].mean().item()*100:>7.1f}")
